# 04 · T5 / BART Mini —— Encoder-Decoder 统一文本到文本 + 去噪

**家族位置**：`05_Transformer_NLP` 第 4 站。01 已拆全量 Transformer，02 只留 Encoder(BERT)，03 只留 Decoder(GPT)，本章**拼回全量**作统一接口：`T5` 用“复制/翻转”文本到文本（与 05-01 同 toy 可比），`BART` 在其上加随机掩码去噪——两者同 Encoder-Decoder，同 `seq2seq` 范式，不同在“输入噪不噪”。无外网，CPU 2 分钟级。

**学习目标**
1. T5 统一范式：一切任务都是 `src→tgt`（复制即翻译的最小原型）
2. BART 去噪：把 25% token 换成 `[MASK]`，仍要还原原句——预训练思想的 toy 演示
3. 与 05-01 同 toy（S=6, vocab=8 复制；S=8 去噪）同台，seq-acc 对比
4. 交叉注意力对角/去噪热力可视化

## 1. 原理：从“复制即翻译”到“带噪复制即去噪”

### 通俗理解

**一句话**：T5 说“所有任务都是翻译”——复制是最小的翻译（`src==tgt`），翻转是带重排的翻译；BART 说“先把原文挖掉几个洞再翻译”，模型要学会“补洞”。

**比喻**：T5 抄作业（照抄），BART 完形填空后抄（被涂掉 25% 再还原）——后者难 20 个点，正好体现“预训练要难才有效”。

### 结构账

```
T5 复制：  src(6) == tgt(6)   → tgt_input=[BOS,tgt0..4]   CE  (05-01 同款)
BART 去噪： src_noisy = 掩码 src 的 25% 为 [MASK]=vocab  →  tgt=原 src   去噪还原
骨架：  TransformerTiny(d32/h4/L2，复用 01 全量)   Encoder 双向 + Decoder 因果 + cross
评估：  seq-acc 整句全对 + 交叉热力（对角/去噪聚焦）、样本条带（噪位红框）
```

- **与 05-01 同台**：同 S=6 vocab=8 复制，数字可直接比 05-01 的 1.0
- **去噪**：S=8 p_mask=0.25，噪位随机，BART 要学“看上下文猜被挖掉的”

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "common").exists():
    ROOT = ROOT.parent
assert (ROOT / "common").exists(), "向上未找到 common 目录"
sys.path.insert(0, str(ROOT))

from common.data import make_copy_data, make_denoise_data
from common.models import TransformerTiny, set_torch_seed
from common.utils import set_seed, setup_chinese_font, plot_attention, seq_accuracy, token_accuracy

set_seed(0)
setup_chinese_font()
FIGS = Path.cwd() / "figs"
FIGS.mkdir(exist_ok=True)
print("torch:", torch.__version__)

# T5 复制：同 05-01 S=6
VOCAB, SEQ_LEN = 8, 6
Xc_all, Yc_all = make_copy_data(800, SEQ_LEN, VOCAB, seed=0)
X_copy, Y_copy = Xc_all[:600], Yc_all[:600]
Xc_te, Yc_te = Xc_all[600:], Yc_all[600:]
# BART 去噪：S=8 更长，25% 掩码
SEQ_LEN_DN = 8
X_noisy, Y_clean, MASK_ID = make_denoise_data(800, SEQ_LEN_DN, VOCAB, mask_id=VOCAB, p_mask=0.25, seed=0)
Xn_noisy_t, Yn_clean_t, _ = make_denoise_data(200, SEQ_LEN_DN, VOCAB, mask_id=VOCAB, p_mask=0.25, seed=1)
print(f"复制 {len(X_copy)} S={SEQ_LEN} vocab={VOCAB} | 去噪 train {len(X_noisy)}/{len(Xn_noisy_t)} S={SEQ_LEN_DN} MASK={MASK_ID} p=0.25")


## 2. 数据：复制 vs 带噪复制

复制 600 条 S=6（05-01 同款）；去噪 800/200 S=8，25% 位置随机换成 [MASK]=8（如 `5 8 7 2 8 0 3 1` 中 8 为被挖洞），目标为原句。

In [ ]:
# fig0：去噪样例条带（原句 vs 噪句，噪位红框）
fig, axes = plt.subplots(2, 1, figsize=(8.5, 2.6), sharex=False)
for ax, title, seq in zip(axes, ["原句 clean", "噪句 noisy ([MASK]=8 红框)"], [Y_clean[0], X_noisy[0]]):
    ax.set_xlim(0, SEQ_LEN_DN); ax.set_ylim(0, 1.4); ax.axis("off")
    ax.set_title(title, fontsize=9, loc="left")
    for j, tok in enumerate(seq):
        is_mask = (tok == MASK_ID)
        col = "#FADBD8" if is_mask else "#D5F5E3"
        ec = "#C0392B" if is_mask else "#1E8449"
        ax.add_patch(plt.Rectangle((j+0.08, 0.45), 0.84, 0.6, facecolor=col, edgecolor=ec, linewidth=1.0))
        ax.text(j+0.5, 0.75, str(tok), ha="center", va="center", fontsize=10, weight="bold", color=ec if is_mask else "black")
plt.suptitle(f"BART 去噪样例（S={SEQ_LEN_DN}，25% 挖洞）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig0_denoise.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. 训练：统一用 TransformerTiny（Encoder-Decoder 全量），两任务同配方

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

def train_eval_seq2seq(model, X, Y, epochs=60, batch=32, lr=8e-3, seed=0, Xte=None, Yte=None):
    VOC = int(max(max(r) for r in Y) + 1)  # vocab
    bos = model.bos
    xt = torch.tensor(X, dtype=torch.long); yt = torch.tensor(Y, dtype=torch.long)
    tgt_in = torch.cat([torch.full((yt.size(0),1), bos, dtype=torch.long), yt[:,:-1]], dim=1)
    loader = DataLoader(TensorDataset(xt, tgt_in, yt), batch_size=batch, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    lossf = nn.CrossEntropyLoss()
    hist = []
    for ep in range(1, epochs+1):
        model.train()
        tot=0
        for xb, ti, yb in loader:
            logits = model(xb, ti)
            loss = lossf(logits.reshape(-1, VOC), yb.reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item()*len(xb)
        hist.append(tot/len(X))
    # eval
    model.eval()
    with torch.no_grad():
        # greedy
        if Xte is not None:
            pred = model.greedy(torch.tensor(Xte, dtype=torch.long), yt.size(1)).tolist()
            sacc = seq_accuracy(pred, Yte); tacc = token_accuracy(pred, Yte)
        else:
            pred = model.greedy(xt[:200], yt.size(1)).tolist()
            sacc = tacc = None
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return hist, sacc, tacc, n_params, pred

# T5 复制：同 05-01
set_torch_seed(0)
t5 = TransformerTiny(VOCAB, d_model=32, n_head=4, n_layer=2, d_ff=64, max_len=32)
h_t5, sacc_t5, tacc_t5, n_t5, pred_t5 = train_eval_seq2seq(t5, X_copy, Y_copy, epochs=60, seed=0, Xte=Xc_te, Yte=Yc_te)
print(f"T5 复制 seq-acc={sacc_t5} tok-acc={tacc_t5} params={n_t5} | （05-01 Transformer 42.5k 双 1.0 可比）")

# BART 去噪：S=8，vocab+1 含 MASK 嵌入，需 vocab+1 的 Transformer（MASK id = VOCAB）
# 复用 TransformerTiny 但 vocab 视作 VOCAB（MASK 在 emb 内为 vocab 行，src 可含 MASK，tgt 不含）
# 技巧：模型 vocab 仍为 VOCAB，MASK 的 emb 行已在 vocab+1 内可学（src 的 MASK id == VOCAB 有嵌入）
set_torch_seed(0)
bart = TransformerTiny(VOCAB, d_model=32, n_head=4, n_layer=2, d_ff=64, max_len=32)
h_bart, sacc_bart, tacc_bart, n_bart, pred_bart = train_eval_seq2seq(bart, X_noisy, Y_clean, epochs=60, seed=0, Xte=Xn_noisy_t, Yte=Yn_clean_t)
print(f"BART 去噪 seq-acc={sacc_bart:.4f} tok-acc={tacc_bart:.4f} params={n_bart} | 25% 挖洞更难")

# fig1：双任务 loss 同台
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(h_t5, label="T5 复制 S=6", color="#4C72B0")
ax.plot(h_bart, label="BART 去噪 S=8 p=0.25", color="#DD8452")
ax.set_xlabel("epoch"); ax.set_ylabel("CE loss")
ax.set_title("T5 复制 vs BART 去噪（同骨架不同难度）")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGS / "fig1_loss.png", dpi=150, bbox_inches="tight")
plt.show()

# fig2：seq-acc 柱状
fig, ax = plt.subplots(figsize=(5, 3.6))
ax.bar(["T5 复制\nS=6", "BART 去噪\nS=8 25%MASK"], [sacc_t5 or 0, sacc_bart], color=["#4C72B0","#DD8452"])
ax.set_ylim(0,1.12); ax.set_ylabel("seq-acc（整句全对）")
for i, v in enumerate([sacc_t5 or 0, sacc_bart]):
    ax.text(i, v+0.03, f"{v:.3f}", ha="center", fontsize=10)
ax.set_title("统一文本到文本：复制满分 vs 去噪更难")
plt.tight_layout()
plt.savefig(FIGS / "fig2_bar.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. 可视化：交叉注意力对角 + 去噪对焦

In [ ]:
# fig3：T5 复制交叉注意力（S=6 对角）— 单句层2头平均
with torch.no_grad():
    idx = 0
    src = torch.tensor([X_copy[idx]], dtype=torch.long)
    enc = t5.encode(src)
    tgt_in = torch.cat([torch.full((1,1), t5.bos, dtype=torch.long), torch.tensor([Y_copy[idx][:-1]], dtype=torch.long)], dim=1)
    S = tgt_in.size(1)
    causal = torch.tril(torch.ones(S, S)).view(1,1,S,S)
    x = t5.pos(t5.emb(tgt_in))
    for li, lyr in enumerate(t5.layers_dec):
        a,_ = lyr["self_attn"](x,x,x,mask=causal); x = lyr["ln1"](x+a)
        b, attn = lyr["cross_attn"](x, enc, enc, mask=None)
        if li==t5.n_layer-1: cross = attn.mean(dim=1)[0].numpy()
        x = lyr["ln2"](x+b); x = lyr["ln3"](x+lyr["ffn"](x))
fig, ax = plt.subplots(figsize=(5.2, 4.4))
plot_attention(ax, cross, X_copy[idx], Y_copy[idx], title="T5 复制 · 交叉注意力（对角线=照抄对齐）")
plt.tight_layout()
plt.savefig(FIGS / "fig3_attn_t5.png", dpi=150, bbox_inches="tight")
plt.show()

# fig4：BART 去噪交叉注意力（S=8，噪位列应被弱化或仍需上下文）
with torch.no_grad():
    idx = 0
    src = torch.tensor([X_noisy[idx]], dtype=torch.long)
    enc = bart.encode(src)
    tgt_in = torch.cat([torch.full((1,1), bart.bos, dtype=torch.long), torch.tensor([Y_clean[idx][:-1]], dtype=torch.long)], dim=1)
    S = tgt_in.size(1)
    causal = torch.tril(torch.ones(S, S)).view(1,1,S,S)
    x = bart.pos(bart.emb(tgt_in))
    for li, lyr in enumerate(bart.layers_dec):
        a,_ = lyr["self_attn"](x,x,x,mask=causal); x = lyr["ln1"](x+a)
        b, attn = lyr["cross_attn"](x, enc, enc, mask=None)
        if li==bart.n_layer-1: cross = attn.mean(dim=1)[0].numpy()
        x = lyr["ln2"](x+b); x = lyr["ln3"](x+lyr["ffn"](x))
fig, ax = plt.subplots(figsize=(5.5, 4.6))
# src 标签：MASK 标红
src_tokens = ["M" if t==MASK_ID else str(t) for t in X_noisy[idx]]
plot_attention(ax, cross, src_tokens, Y_clean[idx], title="BART 去噪 · 交叉注意力（M=被挖洞，需上下文补）")
plt.tight_layout()
plt.savefig(FIGS / "fig4_attn_bart.png", dpi=150, bbox_inches="tight")
plt.show()

# fig5：去噪样本条带（噪句 vs 还原，3 例；噪位红框，错位红字）
with torch.no_grad():
    # 取测试集 3 例
    for k in range(3):
        pass
pred_b = bart.greedy(torch.tensor(Xn_noisy_t[:3], dtype=torch.long), SEQ_LEN_DN).tolist()
fig, axes = plt.subplots(3, 1, figsize=(8.8, 4.2), sharex=False)
for ax, noisy, clean, pred in zip(axes, Xn_noisy_t[:3], Yn_clean_t[:3], pred_b):
    ax.set_xlim(0, SEQ_LEN_DN); ax.set_ylim(0, 2.1); ax.axis("off")
    ok = (pred==clean)
    ax.set_title(f"clean {clean}  noisy {noisy}  pred {pred}  {'✓' if ok else '✗'}  seq-acc {'1' if ok else '0'}", fontsize=8, loc="left", color="#1E8449" if ok else "#C0392B")
    for j, (n_tok, c_tok, p_tok) in enumerate(zip(noisy, clean, pred)):
        is_mask = (n_tok==MASK_ID)
        col = "#FADBD8" if is_mask else "#D5D8DC"
        ec = "#C0392B" if is_mask else "#555"
        ax.add_patch(plt.Rectangle((j+0.06, 1.05), 0.88, 0.55, facecolor=col, edgecolor=ec, linewidth=0.9))
        ax.text(j+0.5, 1.32, str(n_tok) if not is_mask else "M", ha="center", va="center", fontsize=9, color=ec, weight="bold" if is_mask else "normal")
        # 下排：真实/预测
        ax.add_patch(plt.Rectangle((j+0.06, 0.22), 0.88, 0.55, facecolor="#D5F5E3" if p_tok==c_tok else "#FADBD8", edgecolor="#1E8449" if p_tok==c_tok else "#C0392B", linewidth=0.7))
        ax.text(j+0.5, 0.49, str(p_tok), ha="center", va="center", fontsize=9)
        if p_tok!=c_tok:
            ax.text(j+0.5, 0.08, f"true {c_tok}", ha="center", fontsize=6, color="#C0392B")
plt.suptitle("BART 去噪抽样（上：噪句 M=挖洞；下：预测，错位红框）", fontsize=11)
plt.tight_layout()
plt.savefig(FIGS / "fig5_samples.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"BART 测试 3 例 seq-acc demo：{[1 if p==c else 0 for p,c in zip(pred_b, Yn_clean_t[:3])]}")


## 5. 总结与下一步

**本项目收获**

1. T5 统一范式：复制即最简翻译，seq-acc 1.0 与 05-01 同台，交叉对角热力复现
2. BART 去噪：25% 挖洞后仍需还原，seq-acc 低于复制 20 点，难才值得预训练
3. 同骨架同配方双任务，唯一变量“噪不噪”即难度差异，参数 42.5k 量级
4. 衔接 01/02/03：全量 Encoder-Decoder = 02 的 Encoder + 03 的 Decoder + cross，三变体拼成 05 主干；下站做线性 vs 二次拓展对照

**下一步**：`05_Mamba_vs_Transformer_Mini`（拓展对照，SSM 线性复杂度 vs Transformer 二次，与 04-04 共用实现）。